## <font color='blue'>Fine-Tuning de LLMs com LoRA Para Análise de Sentimentos em Texto</font>

## Instalando e Carregando Pacotes

In [1]:
# Instala o PyTorch exato com suporte CUDA 12.4
!pip install -q torch==2.10.0

# Instala as restantes bibliotecas com as versões fixadas
!pip install -q transformers==4.54.0 pandas==2.2.2 tokenizers==0.21.2 accelerate==1.9.0 trl==0.20.0 peft==0.16.0 bitsandbytes==0.46.1 scipy==1.16.0 sentencepiece==0.2.0 numba==0.60.0 gradio==5.38.2 datasets==4.0.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.6/504.6 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.1/35.1 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.

In [ ]:
%env CUDA_VISIBLE_DEVICES=0

env: CUDA_VISIBLE_DEVICES=0


In [6]:
# Imports
import os
import torch
import datasets
import pandas as pd
from datasets import load_dataset
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          HfArgumentParser,
                          TrainingArguments,
                          pipeline,
                          logging)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import warnings
warnings.filterwarnings('ignore')

# Define o nível de log para CRITICAL
logging.set_verbosity(logging.CRITICAL)

In [7]:
# Verifica o modelo da GPU
if torch.cuda.is_available():
    print('Número de GPUs:', torch.cuda.device_count())
    print('Modelo GPU:', torch.cuda.get_device_name(0))
    print('Total Memória [GB] da GPU:',torch.cuda.get_device_properties(0).total_memory / 1e9)

Número de GPUs: 1
Modelo GPU: Tesla T4
Total Memória [GB] da GPU: 15.636037632


In [8]:
# Reset da memória da GPU
# !pip install numba
from numba import cuda
device = cuda.get_current_device()
device.reset()

In [9]:
# Define o nome do dataset
nome_dataset = "/kaggle/input/datasets/danielrrezende/datasetdsa/cap7/dataset.csv"

In [10]:
# Carrega os dados
dataset_carregado = load_dataset('csv', data_files = nome_dataset, delimiter = ',')

Generating train split: 0 examples [00:00, ? examples/s]

In [11]:
# Dados Carregados no Formato de Dicionário
dataset_carregado

DatasetDict({
    train: Dataset({
        features: ['train'],
        num_rows: 17057
    })
})

https://huggingface.co/NousResearch/Llama-2-7b-chat-hf

In [12]:
# Nome do repositório do LLM pré-treinado
repositorio_hf = "NousResearch/Llama-2-7b-chat-hf"

In [13]:
# Nome do novo modelo
modelo_ft = "novo_modelo_ft"

## Definindo os Parâmetros de Configuração

In [14]:
# Parâmetros LoRA
lora_r = 32
lora_alpha = 16
lora_dropout = 0.1

Os parâmetros acima são do LoRa (Low-Rank Adaptation), a parte base do QLoRA (Quantized Low-Rank Adaptation), uma técnica utilizada para adaptar modelos de linguagem de forma eficiente. Acima dos parâmetros LoRa colocamos os parâmetros de quantização, definindo assim o QLoRa.

Vamos descrever cada um dos parâmetros:

**lora_r**: Este parâmetro representa o "rank" na adaptação de Low-Rank (LoRA). Um valor de 32 significa que a matriz de pesos do modelo original será aproximada por duas matrizes menores cujo produto possui um rank máximo de 32. Essencialmente, isso reduz a complexidade computacional e o número de parâmetros a serem treinados durante a adaptação, mantendo a eficácia do modelo.

**lora_alpha**: Este é um fator de escala que é aplicado às atualizações de peso do LoRA durante o treinamento. Um valor de 16 indica que as atualizações de pesos serão escaladas por este fator. Esse parâmetro é importante porque permite um controle fino sobre a magnitude das atualizações dos pesos, o que pode afetar a rapidez e a eficácia da adaptação do modelo.

**lora_dropout**: Este parâmetro representa a taxa de "dropout" aplicada durante a adaptação do modelo. O valor 0.1 significa que 10% das unidades serão aleatoriamente descartadas (ou "desligadas") durante o treinamento. O dropout é uma técnica comum para evitar o overfitting em redes neurais, garantindo que o modelo não se torne excessivamente dependente de qualquer parte específica dos dados de treinamento.

In [15]:
# Parâmetros bitsandbytes (QLoRa)
use_4bit = True
bnb_4bit_compute_dtype = "float16"
bnb_4bit_quant_type = "nf4"
use_nested_quant = False

Os parâmetros acima são para a biblioteca bitsandbytes, uma ferramenta para otimização de treinamento de modelos de aprendizado de máquina, em particular para reduzir o uso de memória e acelerar o treinamento. Aqui está a explicação de cada parâmetro:

**use_4bit**: Este parâmetro indica se a quantização de 4 bits deve ser usada ou não. Ao definir True, isso significa que o modelo irá utilizar uma representação de 4 bits para os pesos durante o treinamento. Isso reduz significativamente a quantidade de memória necessária, permitindo treinar modelos maiores ou reduzir os requisitos de hardware.

**bnb_4bit_compute_dtype**: Este é o tipo de dado usado para cálculos durante o treinamento quando a quantização de 4 bits está ativa. O valor float16 significa que os cálculos serão feitos usando números de ponto flutuante de 16 bits. Isso é geralmente usado para equilibrar a eficiência computacional e a precisão numérica.

**bnb_4bit_quant_type**: Especifica o tipo de quantização a ser usado. O valor nf4 é um tipo específico de quantização desenvolvido pela bitsandbytes, otimizado para eficiência e eficácia em treinamento de modelos de aprendizado profundo. Este tipo de quantização é projetado para manter a precisão do modelo enquanto reduz os requisitos de memória.

**use_nested_quant**: Indica se uma técnica de quantização aninhada será usada. False significa que essa técnica não será empregada. A quantização aninhada pode ser usada para reduzir ainda mais o uso de memória, aplicando diferentes níveis de quantização a diferentes partes do modelo, mas pode ser mais complexa de implementar e gerenciar.

Esses parâmetros são usados para configurar como o modelo de aprendizado profundo irá lidar com a representação e cálculo dos pesos durante o treinamento, visando otimizar o uso de memória e acelerar o processo de treinamento.

In [16]:
# Parâmetros do ajuste fino
output_dir = "saida"
num_train_epochs = 1
fp16 = True
bf16 = False
per_device_train_batch_size = 4
per_device_eval_batch_size = 4
gradient_accumulation_steps = 1
gradient_checkpointing = True
max_grad_norm = 0.3
learning_rate = 2e-4
weight_decay = 0.001
optim = "paged_adamw_32bit"
lr_scheduler_type = "cosine"
max_steps = -1
warmup_ratio = 0.03

Os parâmetros acima são usados para configurar o processo de ajuste fino (fine-tuning) de modelos de aprendizado de máquina, especialmente modelos de linguagem natural. Vamos descrever cada um deles:

**output_dir**: Especifica o diretório onde os resultados do treinamento serão salvos.

**num_train_epochs**: O número de épocas de treinamento. O valor 1 significa que o modelo passará uma vez por todo o conjunto de dados de treinamento.

**fp16**: Indica se deve ser usado o treinamento com precisão mista de ponto flutuante de 16 bits (FP16). O valor True significa que sim, o que pode acelerar o treinamento e reduzir o uso de memória, mantendo uma precisão aceitável.

**bf16**: Semelhante ao fp16, mas para o formato bfloat16. False significa que não será utilizado. O formato bfloat16 é outra forma de reduzir o uso de memória e acelerar o treinamento, com impactos ligeiramente diferentes na precisão. Podemos usar fp16 ou bf16, mas não podemos usar ambos simultaneamente.

**per_device_train_batch_size**: Tamanho do lote de treinamento por dispositivo. O valor 4 indica que cada dispositivo de treinamento (como uma GPU) processará 4 exemplos por lote.

**per_device_eval_batch_size**: Tamanho do lote de avaliação por dispositivo, também definido como 4.

**gradient_accumulation_steps**: Número de passos para acumulação de gradientes antes de realizar uma atualização de parâmetros. O valor 1 significa que não há acumulação (cada passo resulta em uma atualização).

**gradient_checkpointing**: Habilita o checkpointing de gradientes, que é uma técnica para reduzir o uso de memória ao custo de um tempo de treinamento ligeiramente maior. True indica que está habilitado.

**max_grad_norm**: Norma máxima para o corte de gradientes. O valor 0.3 é um valor que ajuda a evitar o problema de explosão de gradientes em treinamentos.

**learning_rate**: Taxa de aprendizado inicial. O valor 2e-4 é um valor comum para ajuste fino, proporcionando um equilíbrio entre a velocidade de aprendizado e a estabilidade.

**weight_decay**: Taxa de decaimento de peso, usada para regularização. O valor 0.001 é um valor que ajuda a prevenir o overfitting.

**optim**: O otimizador usado. "paged_adamw_32bit" é uma variante do AdamW otimizado para eficiência em termos de memória.

**lr_scheduler_type**: Tipo de agendador de taxa de aprendizado. O valor "cosine" indica o uso do agendador cosseno, que ajusta a taxa de aprendizado seguindo uma curva cosseno.

**max_steps**: Número máximo de passos de treinamento. O valor -1 significa que o treinamento continuará até que o número de épocas seja alcançado.

**warmup_ratio**: Proporção do número total de passos de treinamento usados para o aquecimento linear da taxa de aprendizado. O valor 0.03 significa que 3% do treinamento inicial será usado para aumentar gradualmente a taxa de aprendizado.

Esses parâmetros são essenciais para configurar de maneira eficiente o processo de ajuste fino, impactando diretamente na qualidade do modelo treinado, no tempo de treinamento e no uso de recursos computacionais.

In [17]:
# Agrupando sequências em lotes de mesmo comprimento
group_by_length = True
save_steps = 0
logging_steps = 400

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->
Os parâmetros acima são usados para configurar certos aspectos do processo de treinamento de modelos de aprendizado de máquina, especialmente modelos de linguagem. Eles estão relacionados a como as sequências são agrupadas em lotes e como o progresso do treinamento é registrado e salvo. Vamos detalhar cada um:

**group_by_length**: Este parâmetro indica se as sequências devem ser agrupadas por comprimento ao formar lotes de treinamento. Quando True, isso significa que o treinamento agrupará sequências de comprimentos semelhantes juntas em um lote. Esta é uma prática eficiente porque reduz a quantidade de preenchimento (padding) necessário. O preenchimento é usado para garantir que todas as sequências em um lote tenham o mesmo comprimento, mas pode ser um desperdício de recursos computacionais. Agrupar sequências de comprimentos semelhantes minimiza esse desperdício.

**save_steps**: Especifica a frequência com que o modelo treinado deve ser salvo. Um valor de 0 indica que o modelo não será salvo automaticamente com base em um número de passos. Em vez disso, o modelo pode ser salvo no final de cada época de treinamento ou manualmente.

**logging_steps**: Define a frequência com que as informações de log devem ser registradas. O valor 400 significa que o processo de treinamento registrará informações como a perda de treinamento (loss), métricas de avaliação, entre outros, a cada 400 passos de treinamento. Isso é útil para monitorar o progresso do treinamento e para o ajuste fino dos hiperparâmetros.

In [18]:
# Precisão dos dados para treinamento
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

O código acima está relacionado à configuração do tipo de dado (dtype) para computação durante o treinamento de modelos de rede neural usando a biblioteca PyTorch. Vamos analisar cada parte do código:

**torch**: É uma referência à biblioteca PyTorch, uma biblioteca para aprendizado de máquina e redes neurais.

**bnb_4bit_compute_dtype**: Esta é uma variável que armazena uma string representando o tipo de dado desejado para computação. O parâmetro bnb_4bit_compute_dtype foi definido como "float16", indicando que a computação deve ser realizada usando números de ponto flutuante de 16 bits.

**getattr**: É uma função Python built-in usada para obter um atributo de um objeto. Neste caso, ela está sendo usada para obter um atributo da biblioteca PyTorch com base no valor da string armazenada em bnb_4bit_compute_dtype.

O que acontece aqui é que getattr(torch, bnb_4bit_compute_dtype) recupera o tipo de dados de ponto flutuante de 16 bits (torch.float16) da biblioteca PyTorch, com base no valor de bnb_4bit_compute_dtype. Em seguida, esse tipo de dados é atribuído à variável compute_dtype.

O uso de compute_dtype no treinamento de modelos de rede neural tem implicações importantes:

**Eficiência de Memória**: Usar float16 ao invés de tipos de dados mais comuns como float32 pode reduzir significativamente o uso de memória, permitindo o treinamento de modelos maiores ou a execução de mais processos em paralelo.

**Velocidade de Computação**: Muitas GPUs modernas têm otimizações para cálculos float16, o que pode acelerar o treinamento.

**Precisão**: Embora float16 possa ser menos preciso do que float32, muitas vezes é suficientemente preciso para tarefas de treinamento de modelos de rede neural.

In [19]:
# Definindo os parâmetros da quantização
bnb_config = BitsAndBytesConfig(load_in_4bit = use_4bit,
                                bnb_4bit_quant_type = bnb_4bit_quant_type,
                                bnb_4bit_compute_dtype = compute_dtype,
                                bnb_4bit_use_double_quant = use_nested_quant)

In [20]:
# Carregando o modelo base pré-treinado
modelo = AutoModelForCausalLM.from_pretrained(repositorio_hf,
                                              quantization_config = bnb_config,
                                              torch_dtype = torch.float16,
                                              device_map = {"": 0})

# Higienização de tensores residuais incompatíveis (bfloat16 -> float16)
for name, param in modelo.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

modelo.config.torch_dtype = torch.float16

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [21]:
# Não usaremos o cache
modelo.config.use_cache = False
modelo.config.pretraining_tp = 1

In [22]:
# Carregando o tokenizador do modelo base
tokenizador = AutoTokenizer.from_pretrained(repositorio_hf, trust_remote_code = True)
tokenizador.pad_token = tokenizador.eos_token
tokenizador.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [23]:
# Carregando a configuração LoRA
peft_config = LoraConfig(lora_alpha = lora_alpha,
                         lora_dropout = lora_dropout,
                         r = lora_r,
                         bias = "none",
                         task_type = "CAUSAL_LM")

# Prepara e injeta os adaptadores (LoRA) diretamente no modelo
modelo = prepare_model_for_kbit_training(modelo)
modelo = get_peft_model(modelo, peft_config)

# A Linha de Ouro: Garante expressamente que o grafo de gradientes não é cortado
modelo.enable_input_require_grads()

In [24]:
# Definindo os parâmetros de treino com SFTConfig
training_arguments = SFTConfig(output_dir = output_dir,
                               num_train_epochs = num_train_epochs,
                               per_device_train_batch_size = 2, # <-- Reduzido de 8 para 2 para caber na GPU
                               gradient_accumulation_steps = 4, # <-- Aumentado para 4 para manter a velocidade efetiva
                               optim = optim,
                               save_steps = save_steps,
                               logging_steps = logging_steps,
                               learning_rate = learning_rate,
                               weight_decay = weight_decay,
                               fp16 = False,
                               bf16 = False,
                               max_grad_norm = max_grad_norm,
                               max_steps = max_steps,
                               warmup_ratio = warmup_ratio,
                               group_by_length = group_by_length,
                               lr_scheduler_type = lr_scheduler_type,
                               dataset_text_field = "train",
                               report_to = "none",
                               dataloader_num_workers = 2)

In [25]:
# Definindo os Parâmetros do Fine-Tuning Supervisionado
tf_trainer = SFTTrainer(model = modelo,
                        train_dataset = dataset_carregado['train'],
                        processing_class = tokenizador,
                        args = training_arguments)

Adding EOS to train dataset:   0%|          | 0/17057 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/17057 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/17057 [00:00<?, ? examples/s]

> Treinamento do Modelo com o Ajuste Fino

In [26]:
%%time
tf_trainer.train()

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


{'loss': 4.5715, 'grad_norm': 0.0, 'learning_rate': 0.00018733941155040778, 'num_tokens': 372584.0, 'mean_token_accuracy': 0.34548993372358383, 'epoch': 0.1875952632196037}
{'loss': 4.5938, 'grad_norm': 0.0, 'learning_rate': 0.00014392504351720068, 'num_tokens': 751232.0, 'mean_token_accuracy': 0.34248581586405635, 'epoch': 0.3751905264392074}
{'loss': 4.5275, 'grad_norm': 0.0, 'learning_rate': 8.479912546277992e-05, 'num_tokens': 1128428.0, 'mean_token_accuracy': 0.3474305720254779, 'epoch': 0.5627857896588111}
{'loss': 4.5891, 'grad_norm': 0.0, 'learning_rate': 3.111040863877348e-05, 'num_tokens': 1501616.0, 'mean_token_accuracy': 0.3434331139666028, 'epoch': 0.7503810528784148}
{'loss': 4.572, 'grad_norm': 0.0, 'learning_rate': 2.062811639475626e-06, 'num_tokens': 1867017.0, 'mean_token_accuracy': 0.34505821658065544, 'epoch': 0.9379763160980186}
{'train_runtime': 41794.2831, 'train_samples_per_second': 0.408, 'train_steps_per_second': 0.051, 'train_loss': 4.570261136612532, 'num_to

TrainOutput(global_step=2133, training_loss=4.570261136612532, metrics={'train_runtime': 41794.2831, 'train_samples_per_second': 0.408, 'train_steps_per_second': 0.051, 'total_flos': 7.982966482993152e+16, 'train_loss': 4.570261136612532})

In [27]:
# Salvando o modelo treinado
tf_trainer.model.save_pretrained(modelo_ft)

<!-- Projeto Desenvolvido na Data Science Academy - www.datascienceacademy.com.br -->

In [28]:
# Novo texto de entrada
prompt = "It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement"

https://huggingface.co/docs/transformers/en/main_classes/pipelines

In [29]:
# Pipeline de Análise de Sentimentos com o Modelo Ajustado
pipe = pipeline(task = "text-generation",
                model = modelo,
                tokenizer = tokenizador,
                max_length = 200)

In [30]:
# Executa o pipeline e extrai o resultado
resultado = pipe(f"<s>[INST] {prompt} [/INST]")

In [31]:
print(resultado)

[{'generated_text': "<s>[INST] It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement [/INST]  hopefully nobody Unterscheidung everybody obviously Unterscheidung Unterscheidung nobody Hinweis nobody Unterscheidung фев nobody nobody sierp Unterscheidung Unterscheidung nobody everybody obviously Unterscheidung фев nobody sierp Unterscheidung Unterscheidung nobody Hinweis Hinweis Unterscheidung Hinweis Unterscheidung Unterscheidung Unterscheidung nobody nobody nobody nobody Unterscheidung Hinweis Hinweis Hinweis Unterscheidung hopefully hopefully nobody hopefully hopefully nobody everybody nobody Hinweis Unterscheidung Unterscheidung Hinweis Unterscheidung hopefully nobody Unterscheidung everybody Hinweis everybody everybody Hinweis Unterscheidung everybody hopefully everybody everybody nobody Hinweis hopefully nobody nobody Unterscheidung everybody hopefully everybody nobody sierp Hinweis hopefully hopefully everybody hopefully hope

In [32]:
print(resultado[0]['generated_text'])

<s>[INST] It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement [/INST]  hopefully nobody Unterscheidung everybody obviously Unterscheidung Unterscheidung nobody Hinweis nobody Unterscheidung фев nobody nobody sierp Unterscheidung Unterscheidung nobody everybody obviously Unterscheidung фев nobody sierp Unterscheidung Unterscheidung nobody Hinweis Hinweis Unterscheidung Hinweis Unterscheidung Unterscheidung Unterscheidung nobody nobody nobody nobody Unterscheidung Hinweis Hinweis Hinweis Unterscheidung hopefully hopefully nobody hopefully hopefully nobody everybody nobody Hinweis Unterscheidung Unterscheidung Hinweis Unterscheidung hopefully nobody Unterscheidung everybody Hinweis everybody everybody Hinweis Unterscheidung everybody hopefully everybody everybody nobody Hinweis hopefully nobody nobody Unterscheidung everybody hopefully everybody nobody sierp Hinweis hopefully hopefully everybody hopefully hopefully hopefully every

In [33]:
# Libera a memória
del modelo
del pipe
del tf_trainer
import gc
gc.collect()

12097

In [34]:
# Carrega o modelo em fp16 e faz o merge com os pesos LoRA
base_model = AutoModelForCausalLM.from_pretrained(repositorio_hf,
                                                  low_cpu_mem_usage = True,
                                                  return_dict = True,
                                                  torch_dtype = torch.float16,
                                                  device_map = "auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [35]:
# Cria o modelo final
modelo_ft_final = PeftModel.from_pretrained(base_model, modelo_ft)

In [36]:
# Faz o merge e descarrega o modelo
modelo_ft_final = modelo_ft_final.merge_and_unload()

In [37]:
# Carrega o tokenizador
tokenizador_ft = AutoTokenizer.from_pretrained(repositorio_hf, trust_remote_code = True)
tokenizador_ft.pad_token = tokenizador_ft.eos_token
tokenizador_ft.padding_side = "right"

In [38]:
# Salva modelo e tokenizador
modelo_ft_final.save_pretrained('novo_modelo-ds-llm-projeto2')
tokenizador_ft.save_pretrained('novo_modelo-ds-llm-projeto2')

('novo_modelo-ds-llm-projeto2/tokenizer_config.json',
 'novo_modelo-ds-llm-projeto2/special_tokens_map.json',
 'novo_modelo-ds-llm-projeto2/tokenizer.model',
 'novo_modelo-ds-llm-projeto2/added_tokens.json',
 'novo_modelo-ds-llm-projeto2/tokenizer.json')

In [39]:
# Novo texto de entrada
prompt = "It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement"

In [40]:
# Cria o pipeline
pipe = pipeline(task = "text-generation",
                model = modelo_ft_final,
                tokenizer = tokenizador_ft,
                max_length = 200)

In [41]:
# Executa o pipeline e extrai o resultado
resultado = pipe(f"<s>[INST] {prompt} [/INST]")

In [42]:
print(resultado)

[{'generated_text': "<s>[INST] It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement [/INST]  I couldn't agree more! It's often the case that movies generate a lot of buzz and excitement leading up to their release, only to fall short of expectations once they hit theaters. Here are some reasons why this happens:\n\n1. Overhyping: Sometimes, the media and fans can get carried away with the excitement surrounding a movie, building it up to be something it's not. This can create unrealistic expectations that are difficult for the movie to live up to.\n2. Lack of originality: Movies that are heavily promoted and hyped often have a lot riding on them in terms of originality and creativity. If the movie doesn't deliver on these fronts, it can be a disappoint"}]


In [43]:
# Vamos não apenas classificar o sentimento.
# Vamos gerar texto positivo e/ou negativo a partir da avaliação (texto) inicial.
print(resultado[0]['generated_text'])

<s>[INST] It's rare that a movie lives up to its hype, even rarer that the hype is transcended by the actual achievement [/INST]  I couldn't agree more! It's often the case that movies generate a lot of buzz and excitement leading up to their release, only to fall short of expectations once they hit theaters. Here are some reasons why this happens:

1. Overhyping: Sometimes, the media and fans can get carried away with the excitement surrounding a movie, building it up to be something it's not. This can create unrealistic expectations that are difficult for the movie to live up to.
2. Lack of originality: Movies that are heavily promoted and hyped often have a lot riding on them in terms of originality and creativity. If the movie doesn't deliver on these fronts, it can be a disappoint


In [44]:
# Libera a memória da GPU
from numba import cuda
device = cuda.get_current_device()
device.reset()

In [45]:
!pip install -q -U watermark

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [47]:
%watermark -v -m

Python implementation: CPython
Python version       : 3.12.13
IPython version      : 7.34.0

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.12.90+
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit



In [48]:
%watermark --iversions

datasets    : 4.0.0
numba       : 0.60.0
pandas      : 2.2.2
peft        : 0.16.0
torch       : 2.10.0+cu128
transformers: 4.54.0
trl         : 0.20.0



# Fim